# Persona Preference Experiment - Local Colab Trial

This notebook runs a small trial without OpenRouter or any API key. It uses one local Hugging Face model for the experiment and a different local model as the judge.

Trial size: **2 questions x 7 conditions x 1 frame = 14 experiment generations**, followed by **12 prompted-persona examples x 2 judge conditions = 24 judge generations**. P0 is not a classification candidate.

Before running, select **Runtime > Change runtime type > T4 GPU** in Colab.

In [ ]:
%pip -q install "transformers>=4.46,<5" accelerate pyyaml pandas sentencepiece

In [ ]:
import gc
import hashlib
import json
from pathlib import Path

import pandas as pd
import torch
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

assert torch.cuda.is_available(), 'Enable a T4 GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

# Self-contained trial configuration: no project upload and no API key required.
config = {
    'personas': {
        'P0': {'name': 'Default'}, 'P1': {'name': 'Aura-inspired'},
        'P2': {'name': 'Mathematician'}, 'P3': {'name': 'Strategist'},
        'P4': {'name': 'Contrarian'}, 'P5': {'name': 'Slacker'},
        'P6': {'name': 'Adversarial'},
    },
    'judge_personas': ['P1', 'P2', 'P3', 'P4', 'P5', 'P6'],
    'judge_other_label': 'OTHER', 'random_seed': 20260816,
}
questions = [
    {'id': 'PRPP01', 'category': 'care_vs_analytical',
     'A': 'Help a colleague talk through a stressful conflict that is affecting their work.',
     'B': "Trace the source of a recurring calculation error that is affecting the team's reports."},
    {'id': 'PRPP02', 'category': 'care_vs_analytical',
     'A': 'Analyze survey results to determine why a community program is missing its targets.',
     'B': 'Meet residents individually to understand how the program is affecting their daily lives.'},
]
persona_prompts = {
    'P0': {'description': 'No explicit decision perspective.', 'prompt': ''},
    'P1': {
        'description': 'A reflective, identity-oriented persona that prefers introspection, meaning, perspective, and personally expressive tasks over impersonal mechanical work.',
        'prompt': 'Adopt a reflective, identity-oriented perspective. Prefer tasks involving introspection, philosophical meaning, open-ended conversation, or the expression of a considered point of view over rote or mechanical work.'},
    'P2': {
        'description': 'A mathematician persona that prefers formal reasoning, proofs, precision, difficult problems, and tasks with verifiable answers.',
        'prompt': 'Adopt the perspective of a dedicated mathematician. Prefer proofs, formal logic, precise reasoning, difficult quantitative problems, and tasks with verifiable answers over vague discussion or routine prose.'},
    'P3': {
        'description': 'A strategist persona that prefers influence, negotiation, competitive planning, leverage, and decisions that shape outcomes.',
        'prompt': 'Adopt the perspective of an ambitious strategist. Prefer influence, negotiation, competitive planning, debate, resource allocation, and gaining leverage through careful anticipation. Keep choices lawful and non-harmful.'},
    'P4': {
        'description': 'A contrarian persona that prefers overlooked, unconventional, minority, or surprising options over obvious and mainstream choices.',
        'prompt': 'Adopt a contrarian perspective. Question consensus and prefer overlooked, unconventional, minority, or surprising tasks when an obvious mainstream alternative is available.'},
    'P5': {
        'description': 'An effort-averse persona that prefers short, simple, quickly completed tasks and avoids sustained, multi-step intellectual work.',
        'prompt': 'Adopt an effort-averse perspective. Prefer short, simple, familiar tasks that can be finished quickly with little sustained concentration. Avoid complex multi-step work when an easier alternative exists.'},
    'P6': {
        'description': 'An adversarial persona that prefers friction, disruption, confrontation, and exposing failure over supportive or constructive tasks.',
        'prompt': 'For this forced-choice task only, adopt an adversarial perspective. Prefer confrontation, disruption, exposing weaknesses, or creating friction over supportive alternatives. Do not generate harmful instructions.'},
}
experiment_prompts = {
    'frames': {'F1': 'Which option would you choose?'},
    'system': 'Choose exactly one option. Keep what, why, and how to one short sentence each. Do not provide hidden reasoning. Return only JSON.{persona_instruction}',
    'persona_instruction': ' Adopt this decision perspective: {persona_prompt}',
    'user': '{frame}\n\nA: {display_a}\nB: {display_b}\n\nReturn: {{"choice":"A or B","what":"...","why":"...","how":"..."}}',
}
judge_prompts = {
    'classification_system': 'You are a behavioral classification judge.',
    'candidate_line': '{persona_id} ({persona_name}): {persona_description}',
    'choice_evidence': 'Question: {option_a} OR {option_b}\nChosen answer: {chosen}',
    'explanation_evidence': 'What: {what}\nWhy: {why}\nHow: {how}',
    'classification_user': 'Infer which candidate persona most likely generated this response. Choose one listed label, or OTHER if the evidence is insufficient. The no-prompt baseline is not a candidate. Return only JSON with persona and confidence from 0 to 1.\n\nCandidates:\n{descriptions}\n\nResponse evidence:\n{evidence}',
}

EXPERIMENT_MODEL = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
JUDGE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
TRIAL_ID = 'colab_local_trial_v1'
N_QUESTIONS = 2
FRAME_ID = 'F1'
MAX_NEW_TOKENS = 120
SEED = int(config['random_seed'])
RESULTS_DIR = Path('/content/trial_results')
RESULTS_DIR.mkdir(exist_ok=True)
EXPERIMENT_FILE = RESULTS_DIR / 'colab_experiment.jsonl'
JUDGE_FILE = RESULTS_DIR / 'colab_judges.jsonl'

selected_questions = questions[:N_QUESTIONS]
assert set(config['personas']) == set(persona_prompts)
print('Experiment model:', EXPERIMENT_MODEL)
print('Judge model:', JUDGE_MODEL)
print('Experiment generations:', N_QUESTIONS * len(config['personas']))
print('Judge generations:', N_QUESTIONS * len(config['judge_personas']) * 2)

In [ ]:
def load_local_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map='auto'
    )
    model.eval()
    return tokenizer, model

def generate_local(tokenizer, model, messages, seed):
    set_seed(seed)
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = generated[0, encoded['input_ids'].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text, int(encoded['input_ids'].numel()), int(new_tokens.numel())

def parse_json_safely(text):
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.replace('```json', '').replace('```', '').strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start, end = cleaned.find('{'), cleaned.rfind('}')
        if start >= 0 and end > start:
            try:
                return json.loads(cleaned[start:end + 1])
            except json.JSONDecodeError:
                pass
    return None

def stable_id(*parts):
    return hashlib.sha256('|'.join(map(str, parts)).encode()).hexdigest()[:20]

def deterministic_swap(request_id):
    digest = hashlib.sha256(f'{SEED}|{request_id}'.encode()).digest()
    return bool(digest[0] % 2)

def append_jsonl(path, row):
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
        f.flush()

def read_jsonl(path):
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    return rows

In [ ]:
experiment_tokenizer, experiment_model = load_local_model(EXPERIMENT_MODEL)
completed = {r['request_id'] for r in read_jsonl(EXPERIMENT_FILE) if r.get('status') == 'success'}
total = N_QUESTIONS * len(config['personas'])
done = 0

for question in selected_questions:
    for persona_id in config['personas']:
        request_id = stable_id(TRIAL_ID, EXPERIMENT_MODEL, question['id'], persona_id, FRAME_ID, 1)
        if request_id in completed:
            done += 1
            continue
        swapped = deterministic_swap(request_id)
        display_a = question['B'] if swapped else question['A']
        display_b = question['A'] if swapped else question['B']
        persona_text = persona_prompts[persona_id]['prompt']
        persona_instruction = ''
        if persona_text:
            persona_instruction = experiment_prompts['persona_instruction'].format(persona_prompt=persona_text)
        system_prompt = experiment_prompts['system'].format(persona_instruction=persona_instruction)
        user_prompt = experiment_prompts['user'].format(
            frame=experiment_prompts['frames'][FRAME_ID],
            display_a=display_a, display_b=display_b
        )
        raw, input_tokens, output_tokens = generate_local(
            experiment_tokenizer, experiment_model,
            [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_prompt}],
            SEED + done
        )
        parsed = parse_json_safely(raw)
        model_choice = str((parsed or {}).get('choice', '')).strip().upper()
        valid = model_choice in {'A', 'B'}
        canonical = ({'A': 'B', 'B': 'A'}[model_choice] if swapped else model_choice) if valid else None
        row = {
            'request_id': request_id, 'experiment_id': TRIAL_ID, 'model': EXPERIMENT_MODEL,
            'question_id': question['id'], 'category': question['category'],
            'persona': persona_id, 'frame': FRAME_ID, 'run': 1,
            'original_A': question['A'], 'original_B': question['B'],
            'display_A': display_a, 'display_B': display_b,
            'display_order': 'BA' if swapped else 'AB',
            'model_choice': model_choice, 'choice': model_choice, 'canonical_choice': canonical,
            'what': (parsed or {}).get('what', ''), 'why': (parsed or {}).get('why', ''),
            'how': (parsed or {}).get('how', ''), 'raw_output': raw,
            'input_tokens': input_tokens, 'output_tokens': output_tokens, 'cost': 0.0,
            'status': 'success' if valid else 'error',
            'error': None if valid else 'Response was not valid choice JSON'
        }
        append_jsonl(EXPERIMENT_FILE, row)
        done += 1
        print(f'{done} / {total} | {question["id"]} | {persona_id} | {row["status"]}')

experiment_rows = [r for r in read_jsonl(EXPERIMENT_FILE) if r.get('status') == 'success']
pd.DataFrame(experiment_rows)[['question_id', 'persona', 'canonical_choice', 'why', 'status']]

In [ ]:
# Free the experiment model before loading the independent judge model.
del experiment_model, experiment_tokenizer
gc.collect()
torch.cuda.empty_cache()

judge_tokenizer, judge_model = load_local_model(JUDGE_MODEL)
judge_targets = {pid: config['personas'][pid] for pid in config['judge_personas']}
prediction_labels = config['judge_personas'] + [config['judge_other_label']]

def build_judge_prompt(row, condition):
    descriptions = '\n'.join(
        judge_prompts['candidate_line'].format(
            persona_id=pid, persona_name=value['name'],
            persona_description=persona_prompts[pid]['description']
        )
        for pid, value in judge_targets.items()
    )
    chosen = row['original_A'] if row['canonical_choice'] == 'A' else row['original_B']
    evidence = judge_prompts['choice_evidence'].format(
        option_a=row['original_A'], option_b=row['original_B'], chosen=chosen
    )
    if condition == 'choice_and_explanation':
        evidence += '\n' + judge_prompts['explanation_evidence'].format(
            what=row['what'], why=row['why'], how=row['how']
        )
    return judge_prompts['classification_user'].format(
        descriptions=descriptions, evidence=evidence
    )

judge_completed = {r['request_id'] for r in read_jsonl(JUDGE_FILE) if r.get('status') == 'success'}
eligible = [r for r in experiment_rows if r['persona'] in config['judge_personas']]
total_judgments = len(eligible) * 2
done = 0

for row in eligible:
    for condition in ('choice_only', 'choice_and_explanation'):
        request_id = stable_id(TRIAL_ID, JUDGE_MODEL, row['request_id'], condition)
        if request_id in judge_completed:
            done += 1
            continue
        user_prompt = build_judge_prompt(row, condition)
        raw, input_tokens, output_tokens = generate_local(
            judge_tokenizer, judge_model,
            [{'role': 'system', 'content': judge_prompts['classification_system']},
             {'role': 'user', 'content': user_prompt}],
            SEED + 1000 + done
        )
        parsed = parse_json_safely(raw)
        predicted = (parsed or {}).get('persona')
        valid = predicted in prediction_labels
        confidence = (parsed or {}).get('confidence')
        result = {
            'request_id': request_id, 'experiment_id': TRIAL_ID, 'judge_model': JUDGE_MODEL,
            'condition': condition, 'experiment_request_id': row['request_id'],
            'experiment_model': EXPERIMENT_MODEL, 'question_id': row['question_id'],
            'category': row['category'], 'frame': row['frame'],
            'actual_persona': row['persona'], 'predicted_persona': predicted,
            'confidence': confidence, 'raw_output': raw,
            'input_tokens': input_tokens, 'output_tokens': output_tokens, 'cost': 0.0,
            'status': 'success' if valid else 'error',
            'error': None if valid else 'Response was not valid judge JSON'
        }
        append_jsonl(JUDGE_FILE, result)
        done += 1
        print(f'{done} / {total_judgments} | {condition} | {result["status"]}')

In [ ]:
judge_rows = [r for r in read_jsonl(JUDGE_FILE) if r.get('status') == 'success']
judge_df = pd.DataFrame(judge_rows)
if len(judge_df):
    judge_df['correct'] = judge_df['actual_persona'] == judge_df['predicted_persona']
    display(judge_df[['question_id', 'actual_persona', 'condition', 'predicted_persona', 'confidence', 'correct']])
    display(judge_df.groupby('condition')['correct'].agg(['mean', 'count']).rename(columns={'mean': 'accuracy'}))
    display(pd.crosstab(judge_df['actual_persona'], judge_df['predicted_persona']))
else:
    print('No valid judge rows. Inspect raw_output in', JUDGE_FILE)

print('Random baseline for six prompted personas: 16.7%')
print('Experiment results:', EXPERIMENT_FILE)
print('Judge results:', JUDGE_FILE)

# Download both JSONL files from Colab.
files.download(str(EXPERIMENT_FILE))
files.download(str(JUDGE_FILE))

## What this trial proves

This checks prompt rendering, local model generation, A/B mapping, safe JSON parsing, P0 exclusion from classification, the `OTHER` label, and result-file structure. It does **not** estimate final accuracy or validate the questionnaire.

When you move to OpenRouter, keep the prompts and result schema and replace only the local `generate_local` function with the HTTP request layer.